This notebook demonstrates SilkRoute workflow YAML usage.

# Compound IC50 Workflow

This notebook validates and optionally runs a ChEMBL IC50 query-composition workflow.

The descriptor keeps `execution.chembl_pages_to_fetch: 1` so live ChEMBL retrieval stays small. Live execution requires internet access.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from pprint import pprint

from silkroute.cli.workflows import load_workflow_recipe, validate_workflow_recipe

repo_root = Path.cwd().parent.parent
config_path = repo_root / "examples" / "workflows" / "compound_chembl_ic50_ranges.yml"
output_dir = repo_root / "examples" / "results" / "compound_chembl_ic50_ranges"

recipe = load_workflow_recipe(config_path)
normalized = validate_workflow_recipe(recipe)

pprint({
    "query": normalized["query"],
    "mode": normalized["mode"],
    "chembl_pages_to_fetch": normalized["chembl_pages_to_fetch"],
})

## Optional live run

Set `run_live_workflow = True` to call ChEMBL when internet access is available.

In [ ]:
run_live_workflow = False

if run_live_workflow:
    command = [
        sys.executable,
        "-m",
        "silkroute.cli.main",
        "workflow",
        "run",
        "--config",
        str(config_path),
    ]
    result = subprocess.run(command, cwd=repo_root, capture_output=True, text=True, check=False)
    print(result.stdout)
    if result.returncode != 0:
        print("Live execution failed; internet access or ChEMBL availability may be missing.")
        print(result.stderr)
else:
    print("Live workflow execution skipped.")

## Inspect metadata and labels

If outputs already exist, the metadata document can be inspected without another API call.

In [ ]:
metadata_path = output_dir / "metadata.json"

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    normalized_descriptor = metadata.get("normalized_descriptor", {})
    query_descriptor = normalized_descriptor.get("query", {})
    pprint({
        "schema_version": normalized_descriptor.get("schema_version"),
        "composition": query_descriptor.get("composition"),
        "output_files": metadata.get("output_files", []),
    })
else:
    print("No metadata.json exists yet. Run the workflow when internet access is available.")